In [1]:
import sys
sys.path.insert(0,'/mnt/AEA8F340A8F3059D/sportsbet/ai-engine')

In [2]:
from fastapi import APIRouter,HTTPException
from fastapi.responses import StreamingResponse
from agents.supervisor import getAgent
from agents.state import AgentState
from config.constants import INTENT_BET_ADVICE
import json,logging

In [3]:
router=APIRouter()
logger=logging.getLogger(__name__)

In [8]:
@router.get("/bet-advisor/{matchId}")
async def betAdvisorStream(matchId:str,userId:str="",query:str=""):
    agent=getAgent()
    if agent is None:
        raise HTTPException(status_code=500,detail="AI agent not ready")
    async def stream():
        try:
            yield f"data:{json.dumps({'type':'start','matchId':matchId})}\n\n"
            state=AgentState(
                query=query or f"Give betting advice for match {matchId}",
                intent=INTENT_BET_ADVICE,
                confidence=1.0,
                slots={
                    "matchId":matchId,
                },
                context={
                    "matchId":matchId,
                    "pageType":"match"
                },
                user_id=userId
            )
            result=await agent.ainvoke(state)
            output=result.get("output",{})
            yield f"data:{json.dumps({'type':'result','data':output},default=str)}\n\n"
            yield f"data:{json.dumps({'type':'done'})}\n\n"
        except Exception as e:
            logger.error(f"bet advisor error {e}")
            yield f"data:{json.dumps({'type':'error','message':str(e)})}\n\n"
    return StreamingResponse(
        stream(),
        media_type="text/event-stream",
        headers={
            "Cache-Control":"no-cache",
            "Connection":"keep-alive",
        }
    )